# ABSOL quickstart
Array-Based Signal Outlier Locator — simulate a contaminated GMRT band-4 scene, look at it, and reproduce the lstsq hub-recovery feasibility demo (milestone M1).

In [ ]:
import numpy as np, torch, matplotlib.pyplot as plt
from absol.geometry import Array
from absol.utils import load_yaml
array = Array.from_yaml('../configs/array_gmrt.yaml')
print(array.name, array.n_ant, 'antennas,', array.baselines().shape[0], 'baselines')
plt.figure(figsize=(5,5)); plt.scatter(array.enu[:,0]/1e3, array.enu[:,1]/1e3, s=12)
for n,(e,nn,u) in zip(array.names, array.enu): plt.annotate(n,(e/1e3,nn/1e3),fontsize=6)
plt.xlabel('E [km]'); plt.ylabel('N [km]'); plt.title('GMRT (real Antenna.def positions)');

## Simulate one contaminated scene (reduced channels for speed)

In [ ]:
from absol.simulator.scenes import SceneGenerator
sim_cfg = load_yaml('../configs/sim_gmrt_band4.yaml')
small = Array.from_yaml('../configs/array_gmrt.yaml', n_channels=512)
gen = SceneGenerator(sim_cfg, small, seed=3)
scene = gen.sample(stage=2)
print('events:', [(e['name'], round(e['strength_sigma'],1)) for e in scene.meta['events']])
b = int(scene.truth_mask.sum(dim=(1,2)).argmax())
fig, ax = plt.subplots(1, 2, figsize=(12,3.5))
ax[0].imshow(scene.vis[b,:,:,0].abs(), aspect='auto', origin='lower'); ax[0].set_title(f'|V| baseline {b}')
ax[1].imshow(scene.truth_mask[b], aspect='auto', origin='lower'); ax[1].set_title('truth mask');

## Feasibility demo: lstsq antenna aggregation on the GMRT layout
RFI amplitude factorizes as $a_i a_j$ over baselines; least squares on log per-baseline excess recovers the coupled-antenna set (hub/ROC demo, regression test 6).

In [ ]:
from absol.features import lstsq_antenna_scores
pairs = array.baselines(); rng = np.random.default_rng(0)
truth = np.zeros(array.n_ant, bool); truth[rng.choice(array.n_ant, 5, replace=False)] = True
a = np.where(truth, 10.0**(-rng.random(array.n_ant)), 1e-3)
v = a[pairs[:,0]]*a[pairs[:,1]]*np.exp(0.3*rng.standard_normal(len(pairs)))
scores = lstsq_antenna_scores(v, pairs, array.n_ant)
order = np.argsort(-scores)
plt.bar(range(array.n_ant), scores[order], color=['crimson' if t else 'grey' for t in truth[order]])
plt.xticks(range(array.n_ant), [array.names[i] for i in order], rotation=90, fontsize=7)
plt.ylabel('lstsq log-coupling'); plt.title('red = truly coupled');

## Train / calibrate / infer
See README: `absol train`, `absol calibrate --run runs/v0`, `absol infer --ms obs.ms --run runs/v0`.